# Glossary

component-wise boosting
: A gradient boosting variant that fits each base learner separately to the
  negative gradient and selects the best one per iteration. Only the selected
  learner's parameters change in that iteration. See
  [](../methodology/functional_gradient_boosting).

hat matrix
: The matrix $H$ such that $\hat y = Hy$. For OLS, $H = X(X^\top X)^{-1}X^\top$.
  Boosting's hat matrix $B_m = I - \prod_{\ell=1}^m (I - \nu H_{j^\star_\ell})$
  is constructed from the selected learners' damped projections, where
  $H_{j^\star_\ell}$ is the hat matrix of the learner selected at step $\ell$.
  See [](../methodology/functional_gradient_boosting).

P-spline
: A B-spline basis with a difference penalty on adjacent coefficients
  ($\lambda \beta^\top K \beta$, where $K = \Delta_d^\top \Delta_d$ for order-$d$ finite
  differences). Combines smooth interpolation with controllable flexibility
  via `df`. See [](../methodology/penalties_df).

difference penalty
: The matrix $K = \Delta_d^\top \Delta_d$ applied to penalize variability in B-spline
  coefficients. $\Delta_d$ is the finite-difference operator of order $d$ (default
  2: penalizes curvature); the symbols follow the [notation page](notation.ipynb). The null space of $K$ consists of polynomials of
  degree $< d$ (e.g. linear functions for $d=2$). See
  [](../methodology/penalties_df).

null space
: The set of coefficient vectors $\beta$ for which $\beta^\top K \beta = 0$.
  For a second-order difference penalty, the null space comprises linear
  functions (intercept + slope). These are unpenalized. See
  [](../methodology/penalties_df).

GAMLSS
: Generalized Additive Models for Location, Scale and Shape. Extends
  classical GLMs by fitting a separate additive predictor for each
  distribution parameter. See [](../methodology/gamlss_framework) and
  [](../methodology/gamlss).

AFT
: Accelerated Failure Time. A family of survival models where
  $\log T = f(x) + \sigma\varepsilon$ with parametric errors (Weibull,
  log-logistic, lognormal). See [](../methodology/survival).

partial likelihood
: The likelihood contribution from observed events in the Cox proportional
  hazards model, which conditions on the risk set at each event time and
  avoids specifying the baseline hazard. See [](../methodology/survival).

risk set
: At event time $t$, the set of observations still under observation
  ($T_j \ge t$). The partial likelihood denominator sums over the risk set.
  See [](../methodology/survival).

cyclic / non-cyclic
: Two update schemes for GAMLSS boosting. Cyclic fitting visits the distribution
  parameters in a fixed order and can give each its own stopping budget.
  Non-cyclic fitting selects one parameter–learner combination per global
  update. See [](../methodology/gamlss).

PFER
: Per-Family Error Rate — the expected number of falsely selected variables.
  Stability-selection bounds relate this quantity to the selection
  threshold under their stated assumptions. See
  [](../methodology/stability_selection).

degrees of freedom (effective)
: Two quantities carry this name. The degrees of freedom of one base learner,
  the `df` argument, are $\operatorname{tr}(2H_\lambda - H_\lambda H_\lambda^\top)$
  for the learner's penalized hat matrix $H_\lambda$ (the mboost definition);
  mboostrs chooses $\lambda$ so that this trace equals the requested `df`
  (see [](../methodology/penalties_df)). The degrees of freedom of the boosted
  model, for a fixed sequence of selected learners with $\hat y = B_m y$, are
  $\operatorname{tr}(B_m)$; the Gaussian AIC uses this trace. The quantity
  $\operatorname{tr}(2B_m-B_mB_m^\top)$ is the reduction of the noise
  contribution to the expected training residual sum of squares, in units of
  $\sigma^2$. None of these traces includes the cost of choosing the learner
  sequence from the data. See [](../methodology/theory_properties) §5.1.

pseudo-residual
: The negative gradient $u_i = -\partial\rho / \partial f_i$ evaluated at the
  current predictor. Each boosting iteration fits base learners to this
  pseudo-residual by least squares. See
  [](../methodology/functional_gradient_boosting).

offset
: The initial constant prediction $\hat f^{[0]} = \argmin_c \sum w_i \rho(y_i, c)$.
  For Gaussian family this is the weighted mean $\bar y$; for Binomial it is
  half the log-odds of the event rate. See [](../methodology/families).

Gram solver
: The cached Cholesky factorization (a lower-triangular matrix $L$ with $LL^\top$ equal to the factored matrix) of a penalized Gram matrix
  $G = X^\top W X + \lambda K$, used for $O(p^2)$ back-solves per boosting step.
  See [](../implementation/linear_algebra).

LearnerStore
: The learner state (Cholesky factors, design matrices) that a fitted Python
  model keeps after fitting, so that `predict()`, `aic()` and `hatvalues()`
  can run without refitting. See [](../implementation/architecture).

calibrate
: Update nuisance parameters using the current predictions, such as
  Huber's threshold $\delta$ or Weibull's scale $\sigma$. See
  [](../methodology/families).

Kronecker product
: The $X_2 \otimes X_1$ construction used for multinomial models, sharing a
  single covariate design across $K-1$ class predictors. Block-diagonal normal
  equations give $K-1$ independent solves with one factorization. See
  [](../methodology/multinomial).

Breslow method
: The tie-handling method for the Cox partial likelihood in mboost and
  mboostrs. All events that share an event time use the same risk-set
  denominator, the sum over every observation at risk at that time. The Efron
  method, which reduces the denominator for each successive event in a tied
  group and is less biased when ties are frequent, is not implemented. See
  [](../methodology/survival).

LSS
: Location, scale and shape. An LSS (distributional) model gives each
  parameter of the response distribution, such as the mean and the standard
  deviation, its own additive predictor. See GAMLSS.

REML
: Restricted maximum likelihood. A criterion for choosing smoothing
  parameters that maximizes the likelihood of the residuals after the
  unpenalized (null-space) part of the model is removed. `gam_refit` uses it
  to refit the terms of a boosted additive model as one penalized regression
  and returns each term's smoothing parameter, effective degrees of freedom
  and pointwise confidence band. See [](../methodology/penalized_reml).

$m_{\text{stop}}$ (`mstop`)
: The number of boosting iterations. Each iteration adds $\nu$ times the
  selected learner's fit, so a smaller `mstop` gives a more strongly shrunk
  model; it is the main tuning parameter and is usually chosen by
  cross-validation or AIC. See [](../methodology/overview).

step length (`nu`, $\nu$)
: The factor by which each selected learner's fit is multiplied before it is
  added to the predictor (default 0.1). In mboostrs code `nu` always means the
  step length; a distribution parameter that is also called nu in R gamlss is
  unrelated.

`bbs`, `bols`, `btree`
: The three most used base learners. `bbs(x)` is a penalized B-spline smooth
  of `x` (20 inner knots, cubic, second-order difference penalty, `df=4` by
  default). `bols(x)` is a linear (or, for a factor, dummy-coded) effect,
  unpenalized unless `df` or `lambda` is given. `btree(x, ...)` is a
  regression tree on the named variables, a single split (stump) by default.
  See [](../methodology/base_learners).

pinball loss
: The loss for quantile regression at level $\tau$:
  $\rho_\tau(y, f) = \tau (y - f)$ if $y \ge f$ and $(1 - \tau)(f - y)$
  otherwise. Its expected value is minimized by the $\tau$-quantile; lower is
  better.

CQR
: Conformalized quantile regression. A conformal method that fits lower and
  upper quantile models, then widens (or narrows) both by a quantile of their
  errors on a held-out calibration set, which gives finite-sample marginal
  coverage. See [](../methodology/conformal).

scorecard (D1a to D11)
: The `mboostrs-scorecard/2.1` grades a library on twelve dimensions, each
  from 1 to 5 (higher is better) against written anchors: D1a recovery under
  a prescribed specification, D1b discovery from data alone, D2 causal
  humility, D3 explanation quality, D4 importance quantification (effect sizes
  with intervals), D5 iteration speed, D6 robustness, D7 human friendliness,
  D8 agent friendliness, D9 prediction-interval calibration, D10
  reproducibility, D11 predictive validity. An anchor is the written
  description of what a given grade requires. See
  [scorecard dimensions](../benchmarks/overview.ipynb#scorecard-dimensions-at-a-glance).